# SECTION 1: CNN MODEL TRAINING

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os
from copy import deepcopy

# ========== SETUP ==========
SAMPLE_RATE = 22050
BATCH_SIZE = 32
NUM_CLASSES = 5

X = np.load("X_balanced.npy", mmap_mode='r').astype(np.float32)
y = np.load("y_balanced.npy")

class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]

# ========== MODEL DEFINITIONS ==========
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * (X.shape[1] // 4) * (X.shape[2] // 4), 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

# ========== DATA LOADER ==========
def get_data_loaders(split_ratio=(0.8, 0.1, 0.1), batch_size=32):
    # Split using indexes first to avoid loading full tensor
    test_size = split_ratio[2]
    val_size = split_ratio[1] / (split_ratio[0] + split_ratio[1])

    # Just use index-based slicing first
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size, stratify=y_temp, random_state=42)

    # Now convert only the split chunks to tensor
    def to_loader(X_part, y_part):
        X_tensor = torch.tensor(X_part, dtype=torch.float32).unsqueeze(1)
        y_tensor = torch.tensor(y_part, dtype=torch.long)
        return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

    train_loader = to_loader(X_train, y_train)
    val_loader = to_loader(X_val, y_val)
    test_loader = to_loader(X_test, y_test)

    return train_loader, val_loader, test_loader, torch.tensor(y_test, dtype=torch.long)


# ========== TRAIN ==========
def train_model(model, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_type == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_type == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif optimizer_type == "RMSprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_model = deepcopy(model.state_dict())
    best_val_acc = 0
    no_improve_epochs = 0

    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        total_correct = 0
        total_samples = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_correct += (pred.argmax(1) == yb).sum().item()
            total_samples += yb.size(0)
        train_acc = total_correct / total_samples
        train_accuracies.append(train_acc)

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                pred = model(xb)
                all_preds.extend(pred.argmax(1).cpu().numpy())
                all_labels.extend(yb.numpy())
        val_acc = accuracy_score(all_labels, all_preds)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {train_acc*100:.2f}% - Val Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = deepcopy(model.state_dict())
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_model)

    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_accuracies)+1), [a*100 for a in train_accuracies], label="Training Accuracy")
    plt.plot(range(1, len(val_accuracies)+1), [a*100 for a in val_accuracies], label="Validation Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return model, best_val_acc

# ========== EVALUATE ==========
def evaluate_model(model, test_loader, y_true):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            pred = model(xb)
            all_preds.extend(pred.argmax(1).cpu().numpy())

    acc = accuracy_score(y_true, all_preds)
    print(f"✅ Final Test Accuracy: {acc * 100:.2f}%")
    print("\n📊 Classification Report:")
    print(classification_report(y_true, all_preds, target_names=class_names))

    cm = confusion_matrix(y_true, all_preds)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap='Blues')
    plt.grid(False)
    plt.title("Confusion Matrix")
    plt.show()

    return acc


In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 70:15:15")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.7, 0.15, 0.15))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
# E1 experiment: change split, optimizer, or learning rate

print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 70:20:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.7, 0.2, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:


print("\n==========================")
print("🔬 Running: CNN | SGD | 0.01 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.01, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
print("\n==========================")
print("🔬 Running: CNN | RMSprop | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="RMSprop", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.0005 | 60 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.0005, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.0001 | 60 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.0001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 20 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=20)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 40 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=40)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"\n🧪 Final Validation Accuracy: {best_val_acc * 100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:

from copy import deepcopy
import os
import pandas as pd
import json

best_val_acc = 0.0
best_model_state = None
best_optimizer_name = ""
results_list = []


In [ ]:
# Insert this block inside your training loop after validation

# === Check and Save Best Model ===
if val_acc > best_val_acc:
    best_val_acc = val_acc
    best_model_state = deepcopy(model.state_dict())
    best_optimizer_name = optimizer_name

# === Store results for comparison ===
results_list.append({
    "model": "CNN",
    "optimizer": optimizer_name,
    "learning_rate": lr,
    "epochs": epochs,
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
})


In [ ]:

# ================= Final Save Step =================
# Save the best CNN model
best_model_path = f"best_cnn_model_{best_optimizer_name.lower()}.pth"
torch.save(best_model_state, best_model_path)
print(f"✅ Best CNN model saved as {best_model_path} with val acc = {best_val_acc:.4f}")

# Save all results to CSV
df_all_results = pd.DataFrame(results_list)
df_all_results.to_csv("cnn_training_results.csv", index=False)
print("📁 All training results saved to cnn_training_results.csv")

# Save all results to JSON
with open("cnn_training_results.json", "w") as json_file:
    json.dump(results_list, json_file, indent=4)
print("📁 All training results saved to cnn_training_results.json")
